In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import scipy.stats as stats

In [5]:
pip install optuna

Note: you may need to restart the kernel to use updated packages.


In [7]:
df1= pd.read_csv("../filtered_data_v1.3.csv")

# Remove extraneous features
df1.drop(columns=['Unnamed: 0', 'Unnamed: 0.1', 'SEQN_new'], inplace=True)

In [4]:
df2= df1.copy()

#clean the noisy data  in target coloumns
df2['MCQ010'] = df2['MCQ010'].apply(lambda x: 1 if x == 1 else 0)
df2['CVD_combined'] = df2['CVD_combined'].apply(lambda x: 1 if x > 1 else x)

# Check class distribution
asthmaclass_counts = df2['MCQ010'].value_counts(normalize=True)
CVDclass_counts = df2['CVD_combined'].value_counts(normalize=True)

print ("Asthma Class Distribution:")
print(asthmaclass_counts)

print("\nCVD Class Distribution:")
print(CVDclass_counts)

Asthma Class Distribution:
MCQ010
0    0.865735
1    0.134265
Name: proportion, dtype: float64

CVD Class Distribution:
CVD_combined
0    0.919196
1    0.080804
Name: proportion, dtype: float64


Asthma Target

In [ ]:
# Remove target (Asthma diagnosis) from features
X = df2.drop(columns=['MCQ010'])
y = df2['MCQ010']

In [ ]:
from sklearn.model_selection import train_test_split

# First split: train 80%, temp 20%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size=0.8, stratify=y, random_state=35)

# Second split: validation 10%, test 10%
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, train_size=0.5, stratify=y_temp, random_state=35)

X_train.shape, X_val.shape, X_test.shape

((115330, 38), (14416, 38), (14417, 38))

In [6]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

In [16]:
# Compute class-balanced sample weights
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Define a shallow decision tree as base estimator
base_estimator = DecisionTreeClassifier(max_depth=5)

# Build AdaBoost model
model = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=100,
    random_state=25
)

# Train the model with class weighting
model.fit(X_train, y_train, sample_weight=sample_weights)

# Predict on validation set
y_pred = model.predict(X_val)

# Print classification report
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.91      0.63      0.75     12481
           1       0.20      0.60      0.30      1935

    accuracy                           0.63     14416
   macro avg       0.56      0.62      0.52     14416
weighted avg       0.82      0.63      0.69     14416



In [ ]:
import optuna
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, average_precision_score

def objective(trial):
    # Hyperparameter search space
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.1, 1.0)
    max_depth = trial.suggest_int('max_depth', 1, 5)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

    # Create base estimator with trial parameters
    estimator_ada = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split
    )

    # Create AdaBoost classifier
    model = AdaBoostClassifier(
        estimator= estimator_ada,
        n_estimators= n_estimators,
        learning_rate=learning_rate,
        random_state=25
    )

    # Compute weights
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    # Fit model
    model.fit(X_train, y_train, sample_weight=sample_weights)

    # Predict and return F1 score for minority class
    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred, pos_label=1)

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50,show_progress_bar=True)

# Best parameters
print("Best params:", study.best_params)

[I 2025-05-22 02:26:46,861] A new study created in memory with name: no-name-017ef36e-9ea6-4f24-912b-de96c33ad658


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-22 02:27:16,631] Trial 0 finished with value: 0.29535311105276973 and parameters: {'n_estimators': 59, 'learning_rate': 0.6579825601856558, 'max_depth': 2, 'min_samples_split': 12}. Best is trial 0 with value: 0.29535311105276973.
[I 2025-05-22 02:29:19,302] Trial 1 finished with value: 0.3036582246098746 and parameters: {'n_estimators': 127, 'learning_rate': 0.46379777852613524, 'max_depth': 4, 'min_samples_split': 7}. Best is trial 1 with value: 0.3036582246098746.
[I 2025-05-22 02:31:55,685] Trial 2 finished with value: 0.2987361164304864 and parameters: {'n_estimators': 215, 'learning_rate': 0.7329446241049545, 'max_depth': 3, 'min_samples_split': 16}. Best is trial 1 with value: 0.3036582246098746.
[I 2025-05-22 02:33:27,755] Trial 3 finished with value: 0.3036690085870414 and parameters: {'n_estimators': 77, 'learning_rate': 0.6928789443793458, 'max_depth': 5, 'min_samples_split': 6}. Best is trial 3 with value: 0.3036690085870414.
[I 2025-05-22 02:36:17,793] Trial 4 f

In [15]:
# Retraining using the best parametres from Optuna

# Combine training and validation sets
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

# Compute class-balanced sample weights
sample_weights = compute_sample_weight(class_weight='balanced', y=y_trainval)

# Use best parameters from Optuna
best_params = {
    'n_estimators': 150,
    'learning_rate': 0.18777180374970204,
    'max_depth': 5,
    'min_samples_split': 4
}

# Step 4: Create base estimator and AdaBoost model
base_estimator = DecisionTreeClassifier(
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split']
)

model = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=best_params['n_estimators'],
    learning_rate=best_params['learning_rate'],
    random_state=25
)

# Train model
model.fit(X_trainval, y_trainval, sample_weight=sample_weights)

# Predict hard labels
y_pred_test = model.predict(X_test)

# Predict probabilities (needed for AUC)
y_prob_test = model.predict_proba(X_test)[:, 1]

# Correct scoring
print("Classification Report:\n", classification_report(y_test, y_pred_test))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_test))
print("PR-AUC:", average_precision_score(y_test, y_prob_test))


Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.63      0.74     12481
           1       0.20      0.61      0.30      1936

    accuracy                           0.63     14417
   macro avg       0.56      0.62      0.52     14417
weighted avg       0.82      0.63      0.69     14417

ROC-AUC: 0.6716080756799923
PR-AUC: 0.25634476319127897


CVD Target

In [8]:
from sklearn.model_selection import train_test_split

# Reload dataset
df3= df2.copy()

# Target cleaning
df3['CVD_combined'] = df3['CVD_combined'].apply(lambda x: 1 if x > 0 else 0)

# Split features and target
X = df3.drop(columns=['CVD_combined','MCQ160C', 'MCQ160B', 'MCQ160E'])
y = df3['CVD_combined']

# Train/test/validation split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=55, stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=55, stratify=y_temp)

X_train.shape, X_val.shape, X_test.shape

((115330, 35), (14417, 35), (14416, 35))

In [10]:
# Compute class-balanced sample weights
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Define a shallow decision tree as base estimator
base_estimator = DecisionTreeClassifier(max_depth=5)

# Build AdaBoost model
model = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=100,
    random_state=25
)

# Train the model with class weighting
model.fit(X_train, y_train, sample_weight=sample_weights)

# Predict on validation set
y_pred = model.predict(X_val)

# Print classification report
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.76      0.85     13252
           1       0.23      0.81      0.36      1165

    accuracy                           0.76     14417
   macro avg       0.60      0.79      0.60     14417
weighted avg       0.92      0.76      0.81     14417



In [13]:
# Combine train + val for final training
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

# Compute sample weights to address class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_trainval)

# Define base estimator
base_tree = DecisionTreeClassifier(max_depth=2)

# Create AdaBoost model
adaboost_model = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=100,
    learning_rate=1.0,
    random_state=42
)

# Train model
adaboost_model.fit(X_trainval, y_trainval, sample_weight=sample_weights)

# Predict and evaluate
y_pred = adaboost_model.predict(X_test)
y_prob = adaboost_model.predict_proba(X_test)[:, 1]

# Evaluation
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.74      0.84     13251
           1       0.22      0.83      0.34      1165

    accuracy                           0.74     14416
   macro avg       0.60      0.78      0.59     14416
weighted avg       0.92      0.74      0.80     14416

ROC-AUC: 0.8545277820153181
PR-AUC: 0.31078994941141835


In [11]:
import optuna
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, average_precision_score

def objective(trial):
    # Hyperparameter search space
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.1, 1.0)
    max_depth = trial.suggest_int('max_depth', 1, 5)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

    # Create base estimator with trial parameters
    estimator_ada = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split
    )

    # Create AdaBoost classifier
    model = AdaBoostClassifier(
        estimator= estimator_ada,
        n_estimators= n_estimators,
        learning_rate=learning_rate,
        random_state=25
    )

    # Compute weights
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    # Fit model
    model.fit(X_train, y_train, sample_weight=sample_weights)

    # Predict and return F1 score for minority class
    y_pred = model.predict(X_val)
    return f1_score(y_val, y_pred, pos_label=1)

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50,show_progress_bar=True)

# Best parameters
print("Best params:", study.best_params)

[I 2025-05-22 13:29:26,911] A new study created in memory with name: no-name-ea380400-84d9-425b-862d-48b880b81262


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-22 13:30:32,948] Trial 0 finished with value: 0.33523021210553544 and parameters: {'n_estimators': 118, 'learning_rate': 0.4804718259622184, 'max_depth': 2, 'min_samples_split': 8}. Best is trial 0 with value: 0.33523021210553544.
[I 2025-05-22 13:30:53,626] Trial 1 finished with value: 0.3306044243163212 and parameters: {'n_estimators': 70, 'learning_rate': 0.7307751177590258, 'max_depth': 1, 'min_samples_split': 10}. Best is trial 0 with value: 0.33523021210553544.
[I 2025-05-22 13:33:14,250] Trial 2 finished with value: 0.3379510860270224 and parameters: {'n_estimators': 254, 'learning_rate': 0.15578675902565, 'max_depth': 2, 'min_samples_split': 11}. Best is trial 2 with value: 0.3379510860270224.
[I 2025-05-22 13:34:35,085] Trial 3 finished with value: 0.3367049545688325 and parameters: {'n_estimators': 148, 'learning_rate': 0.4822610126736451, 'max_depth': 2, 'min_samples_split': 18}. Best is trial 2 with value: 0.3379510860270224.
[I 2025-05-22 13:35:38,603] Trial 4 f

In [15]:
# Retraining using the best parametres from Optuna

# Merge train + val for final training
X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

# Compute sample weights
sample_weights = compute_sample_weight(class_weight='balanced', y=y_trainval)

# Optuna best parameters
best_params = {
    'n_estimators': 219,
    'learning_rate': 0.9027124657817157,
    'max_depth': 5,
    'min_samples_split': 14
}

# Build base estimator and AdaBoost model
base_tree = DecisionTreeClassifier(
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split']
)

model = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=best_params['n_estimators'],
    learning_rate=best_params['learning_rate'],
    random_state=42
)

# Fit model
model.fit(X_trainval, y_trainval, sample_weight=sample_weights)

# Predict
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluation
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))


Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.75      0.85     13251
           1       0.22      0.81      0.35      1165

    accuracy                           0.76     14416
   macro avg       0.60      0.78      0.60     14416
weighted avg       0.92      0.76      0.81     14416

ROC-AUC: 0.8613452446539785
PR-AUC: 0.30739646439435675
